# Gas forward curves — EDA

Interactive companion to `scripts/run_eda.py`. Markets: TTF, THE, JKM. Tenors: M1–M36. Trade dates: 2021-01-04 → 2026-05-11.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path().resolve().parent if pathlib.Path().resolve().name == 'notebooks' else pathlib.Path().resolve()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid', context='talk')
TENORS = [f'M{i}' for i in range(1, 37)]
wide = pd.read_csv(ROOT / 'ml_wide.csv', parse_dates=['TRADE_DATE'])
long = pd.read_csv(ROOT / 'ml_long.csv', parse_dates=['TRADE_DATE', 'DELIVERY_MONTH'])
print(wide.shape, long.shape)
wide.head()

## 1. Per-market basics

In [ ]:
wide.groupby('MARKET')[TENORS].describe().T.head(20)

In [ ]:
mean_curves = wide.groupby('MARKET')[TENORS].mean()
fig, ax = plt.subplots(figsize=(12, 5))
for m in mean_curves.index:
    ax.plot(range(1, 37), mean_curves.loc[m].values, marker='o', label=m, linewidth=2)
ax.set_xlabel('Tenor (M)'); ax.set_ylabel('Mean price'); ax.legend(title='Market'); ax.set_title('Average forward curve')
plt.tight_layout(); plt.show()

## 2. Front-month time series — the macro context

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for m, g in wide.groupby('MARKET'):
    g = g.sort_values('TRADE_DATE')
    ax.plot(g['TRADE_DATE'], g['M1'], label=m, linewidth=1.2)
ax.set_title('M1 across time — 2022 EU gas crisis visible'); ax.legend(); plt.tight_layout(); plt.show()

## 3. Curve heatmap (one market)

In [ ]:
mkt = 'TTF'
sub = wide[wide['MARKET'] == mkt].sort_values('TRADE_DATE')
fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(sub[TENORS].values.T, aspect='auto', origin='lower', cmap='viridis')
ax.set_xlabel('Trade date idx'); ax.set_ylabel('Tenor'); ax.set_title(f'{mkt}: prices'); fig.colorbar(im, ax=ax); plt.show()

## 4. Shape features and regime

In [ ]:
shape = wide[['TRADE_DATE','MARKET']].copy()
shape['LEVEL'] = wide[TENORS].mean(axis=1)
shape['SLOPE_1Y'] = wide['M12'] - wide['M1']
shape['CURVATURE'] = (wide['M1'] + wide['M24'] - 2*wide['M12']) / 2
shape['REGIME'] = np.where(shape['SLOPE_1Y'] >= 0, 'Contango', 'Backwardation')
fig, axes = plt.subplots(1,2, figsize=(14,4))
for ax,col,t in zip(axes,['SLOPE_1Y','CURVATURE'],['M12-M1','(M1+M24)/2 - M12']):
    for mkt in shape['MARKET'].unique():
        sns.kdeplot(shape.loc[shape['MARKET']==mkt, col], ax=ax, label=mkt, fill=True, alpha=0.3)
    ax.set_title(t); ax.legend()
plt.tight_layout(); plt.show()
shape.groupby(['MARKET','REGIME']).size().unstack(fill_value=0)

## 5. PCA — how compressible is shape?

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
fig, ax = plt.subplots(figsize=(10,5))
for mkt in shape['MARKET'].unique():
    X = wide.loc[wide['MARKET']==mkt, TENORS].values
    Xn = StandardScaler().fit_transform(X / X.mean(axis=1, keepdims=True))
    p = PCA().fit(Xn)
    ax.plot(range(1, len(p.explained_variance_ratio_)+1), np.cumsum(p.explained_variance_ratio_), marker='o', label=mkt)
ax.axhline(0.95, color='grey', ls='--'); ax.axhline(0.99, color='k', ls=':')
ax.set_xlim(0,12); ax.set_xlabel('# PCs'); ax.set_ylabel('Cumulative var'); ax.legend(); plt.show()

## 6. After training: inspect AE anomalies
Run `scripts/train_all.py` and `scripts/detect_anomalies.py` first, then load the leaderboard:

In [ ]:
lb_path = ROOT / 'results/anomalies/leaderboard.csv'
if lb_path.exists():
    lb = pd.read_csv(lb_path)
    display(lb)
else:
    print('Run scripts/train_all.py and scripts/detect_anomalies.py first.')

In [ ]:
model_name = 'Conv1d_lat8'
p = ROOT / f'results/anomalies/{model_name}_curve_scores.csv'
if p.exists():
    s = pd.read_csv(p, parse_dates=['TRADE_DATE'])
    fig, ax = plt.subplots(figsize=(14,4))
    for m, g in s.groupby('MARKET'):
        ax.plot(g.sort_values('TRADE_DATE')['TRADE_DATE'], g.sort_values('TRADE_DATE')['score'], label=m, alpha=0.7)
    ax.set_title(f'{model_name} — anomaly score through time'); ax.legend(); plt.show()
    s.sort_values('score', ascending=False).head(15)